# Unified Model Comparison Framework

**Date**: January 23, 2026

## Methodology (per Jan 22, 2026 Resolution)

This notebook implements a **unified experimental setup** for fair cross-model comparison:

1. **Target Variable**: All models predict **normalized DVOL levels** (then denormalized for evaluation)
2. **Preprocessing**: All models use **720-hour rolling window normalization** for both features AND target
3. **Models to Compare**: HAR-RV, OLS, Random Forest, XGBoost, LSTM (rolling, jump-aware)
4. **Rationale**: Structural breaks in DVOL require rolling normalization; features/target must be aligned

### Key References
- Clements & Hendry (1999): Comparing models on different transformations compares incompatible forecasts
- Lim & Zohren (2021): Normalization with sliding windows maintains stationarity for deep learning
- *Risks* (2024): Rolling window models outperform expanding window under structural breaks

### Critical Design Note
**Why normalize the target?** When features are normalized (mean=0, std=1) but the target is raw DVOL with regime-dependent mean, linear models learn a fixed intercept that fails when the regime shifts. By normalizing both features AND target, we ensure alignment and enable fair comparison with LSTM models.

In [58]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load data
DATA_PATH = '/home/lrud1314/PROJECTS_WORKING/THESIS 2025/data/processed/bitcoin_lstm_features_v1.6_final.csv'
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

In [59]:
# Column groups
base_features = ['dvol', 'dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d', 
                 'network_activity', 'nvrv', 'dvol_rv_spread', 'transaction_volume']
jump_features = ['lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d']

## Cell 3: 720-Hour Rolling Window Normalization

In [60]:
# =============================================================================
# PREPROCESSING: 720-Hour Rolling Window Normalization + Train/Val/Test Split
# =============================================================================

# Configuration constants
DEFAULT_NORMALIZATION_WINDOW = 720  # hours (30 days) - Chosen based on literature review
TRAIN_SPLIT_RATIO = 0.60              # 60% for training
VAL_SPLIT_RATIO = 0.20                # 20% for validation
TEST_SPLIT_RATIO = 0.20              # 20% for testing

def apply_rolling_normalization(df, feature_cols, window=720):
    """Apply 720-hour rolling window z-score normalization.
    
    Rationale: 30-day window captures monthly patterns while maintaining adaptivity
    to regime changes in Bitcoin volatility (see Lim & Zohren, 2021).
    """
    df_norm = df.copy()
    scaling_params = {}
    
    for col in feature_cols:
        if col in ['lee_mykland_jump', 'jump_indicator']:  # Handle both naming conventions
            df_norm[col] = df[col]
            continue
        rolling_mean = df[col].rolling(window=window, min_periods=1).mean()
        rolling_std = df[col].rolling(window=window, min_periods=1).std().replace(0, 1)
        df_norm[f'{col}_norm'] = (df[col] - rolling_mean) / rolling_std
        scaling_params[col] = {'mean': rolling_mean.iloc[-1], 'std': rolling_std.iloc[-1]}
    
    df_norm['dvol_rolling_mean'] = df['dvol'].rolling(window=window, min_periods=1).mean()
    df_norm['dvol_rolling_std'] = df['dvol'].rolling(window=window, min_periods=1).std().replace(0, 1)
    df_norm['timestamp'] = df['timestamp']
    return df_norm, scaling_params

# Apply normalization
base_features = ['dvol', 'dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d',
                 'transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread']
jump_features = ['lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d']
all_features = base_features + jump_features

df_norm, scaling_params = apply_rolling_normalization(df, all_features, window=DEFAULT_NORMALIZATION_WINDOW)

# Train/val/test split (60/20/20)
# Rationale: Provides sufficient validation data while maximizing training samples
n_train = int(len(df_norm) * TRAIN_SPLIT_RATIO)
n_val = int(len(df_norm) * VAL_SPLIT_RATIO)

train_df = df_norm.iloc[:n_train].copy()
val_df = df_norm.iloc[n_train:n_train + n_val].copy()
test_df = df_norm.iloc[n_train + n_val:].copy()

## Why Normalizing Lagged Variables is Valid

### The Mathematical Insight

Each lagged feature is a **distinct time series** with its own statistical properties:

- `dvol_lag_1d[t]` = dvol[t-24] → 24-hour delayed series
- `dvol_lag_7d[t]` = dvol[t-168] → 7-day delayed series  
- `dvol[t]` = current series

These are **three different data distributions**, each requiring its own normalization parameters.

### The Key Question

*"Why not normalize `dvol`, then shift the result to get `dvol_lag_1d_norm`?"*

Because normalization must respect the **temporal ordering** — at time t, we can only use data up to time t-1.

### The Math

At time t, the normalized lagged value is:

$$dvol\\_lag\\_1d\\_norm[t] = \\frac{dvol[t-24] - \\mu_{lag1d}(t)}{\\sigma_{lag1d}(t)}$$

where $\\mu_{lag1d}(t)$ and $\\sigma_{lag1d}(t)$ are computed from `dvol_lag_1d[t-719:t-1]` — **the 720-hour window ending at t-1, not t**.

This means each lagged feature is normalized against its **own local history**, preserving the information: *"how unusual is this value relative to recent values of this same lag horizon?"*

### Academic References

| Concept | Full Citation | Key Finding |
|---------|----------------|-------------|
| Rolling window for structural breaks | Chung, V., Espinoza, J., & Quispe, R. (2025). "Forecasting Financial Volatility Under Structural Breaks: A Comparative Study of GARCH Models and Deep Learning Techniques." *Journal of Risk and Financial Management*, 18(9), 494. DOI: 10.3390/jrfm18090494 | "Rolling window estimation...mitigates adverse effects of structural breaks" |
| Sliding window normalization | Lim, B., & Zohren, S. (2021). "Time-series forecasting with deep learning: a survey." *Philosophical Transactions of the Royal Society A: Mathematical, Physical and Engineering Sciences*, 379(2194), 20200093. DOI: 10.1098/rsta.2020.0093 | "Normalization...is a critical preprocessing step for neural networks...standard practice involves scaling data, often utilizing sliding windows" |
| Incomparable forecasts | Clements, M. P., & Hendry, D. F. (1999). *Forecasting Non-stationary Economic Time Series*. The MIT Press. | "Forecasting approaches that apply different transformations are effectively forecasting different data generating processes" |

**Bottom line**: Independent normalization of lagged features is necessary — each lag horizon captures distinct temporal dynamics that must be preserved for accurate multi-scale forecasting.


In [61]:
# =============================================================================
# LINEAR MODELS: SETUP AND DATA PREPARATION
# =============================================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Feature sets
market_features = ['transaction_volume_norm', 'network_activity_norm', 'nvrv_norm', 'dvol_rv_spread_norm']
core_features = ['dvol_lag_1d_norm', 'dvol_lag_7d_norm', 'dvol_lag_30d_norm', 
                 'transaction_volume_norm', 'network_activity_norm', 'nvrv_norm', 'dvol_rv_spread_norm']
har_rv_features = ['dvol_lag_1d_norm', 'dvol_lag_7d_norm', 'dvol_lag_30d_norm']
jump_feature_cols = ['lee_mykland_jump', 'jump_magnitude_norm', 'days_since_jump_norm', 'jump_cluster_7d_norm']

# Union of all features for consistent samples
all_features = list(set(market_features + core_features + jump_feature_cols))

def directional_accuracy(y_true, y_pred):
    """
    Mean Directional Accuracy (MDA) per Pesaran & Timmermann (1992).

    Compares: sgn(A_t - A_{t-1}) vs sgn(F_t - A_{t-1})

    Where:
        A_t = actual value at time t
        F_t = forecast value for time t
        sgn() = sign function

    References:
        Pesaran, M. & Timmermann, A. (1992). "A simple nonparametric test of
        predictive performance". Journal of Business & Economic Statistics, 10(4), 461-465.
    """
    # Align arrays: y_pred[i] forecasts y_true[i+1]
    # We compare: sgn(y_true[i+1] - y_true[i]) vs sgn(y_pred[i] - y_true[i])
    n = len(y_true) - 1  # Need pairs for diff

    # Actual direction: sgn(A_t - A_{t-1})
    actual_direction = np.sign(y_true[1:] - y_true[:-1])

    # Predicted direction: sgn(F_t - A_{t-1})
    # y_pred[i] is forecast for y_true[i+1], made at time i when y_true[i] was known
    predicted_direction = np.sign(y_pred[:-1] - y_true[:-1])

    # Count correct predictions (excluding zero changes)
    valid = (actual_direction != 0)
    correct = (actual_direction[valid] == predicted_direction[valid])

    return (correct.sum() / valid.sum() * 100) if valid.sum() > 0 else 0.0

def prepare_data_splits(train_df, val_df, test_df, feature_cols):
    """Prepare consistent train/val/test splits."""
    y_train = train_df['dvol_norm'].shift(-1)
    y_val = val_df['dvol_norm'].shift(-1)
    y_test = test_df['dvol_norm'].shift(-1)
    
    # Store actual DVOL for directional accuracy
    actual_dvol_train = train_df['dvol'].shift(-1)
    actual_dvol_val = val_df['dvol'].shift(-1)
    actual_dvol_test = test_df['dvol'].shift(-1)
    
    rolling_mean_train = train_df['dvol_rolling_mean'].shift(-1)
    rolling_mean_val = val_df['dvol_rolling_mean'].shift(-1)
    rolling_mean_test = test_df['dvol_rolling_mean'].shift(-1)
    rolling_std_train = train_df['dvol_rolling_std'].shift(-1)
    rolling_std_val = val_df['dvol_rolling_std'].shift(-1)
    rolling_std_test = test_df['dvol_rolling_std'].shift(-1)
    
    X_train_all = train_df[feature_cols].copy()
    X_val_all = val_df[feature_cols].copy()
    X_test_all = test_df[feature_cols].copy()
    
    valid_train = (~y_train.isna()) & (~X_train_all.isna().any(axis=1)) & (~rolling_mean_train.isna())
    valid_val = (~y_val.isna()) & (~X_val_all.isna().any(axis=1)) & (~rolling_mean_val.isna())
    valid_test = (~y_test.isna()) & (~X_test_all.isna().any(axis=1)) & (~rolling_mean_test.isna())
    
    y_train = y_train[valid_train]; y_val = y_val[valid_val]; y_test = y_test[valid_test]
    actual_dvol_train = actual_dvol_train[valid_train]
    actual_dvol_val = actual_dvol_val[valid_val]
    actual_dvol_test = actual_dvol_test[valid_test]
    X_train_all = X_train_all[valid_train]; X_val_all = X_val_all[valid_val]; X_test_all = X_test_all[valid_test]
    
    return (X_train_all, X_val_all, X_test_all, y_train, y_val, y_test,
            {'mean': rolling_mean_train[valid_train].values, 'std': rolling_std_train[valid_train].values},
            {'mean': rolling_mean_val[valid_val].values, 'std': rolling_std_val[valid_val].values},
            {'mean': rolling_mean_test[valid_test].values, 'std': rolling_std_test[valid_test].values},
            {'train': actual_dvol_train.values, 'val': actual_dvol_val.values, 'test': actual_dvol_test.values})

def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, roll_train, roll_val, roll_test, actual_dvol):
    """Evaluate model on train/val/test splits."""
    results = {}
    for name, X, y_true, stats in [('train', X_train, y_train, roll_train), ('val', X_val, y_val, roll_val), ('test', X_test, y_test, roll_test)]:
        y_pred_norm = model.predict(X)
        y_pred_denorm = y_pred_norm * stats['std'] + stats['mean']
        y_true_denorm = y_true.values * stats['std'] + stats['mean']
        
        # Directional accuracy on actual DVOL
        y_actual = actual_dvol[name]
        dir_acc = directional_accuracy(y_actual, y_pred_denorm)
        
        results[name] = {
            'R2_norm': r2_score(y_true, y_pred_norm), 'RMSE_norm': np.sqrt(mean_squared_error(y_true, y_pred_norm)),
            'MAE_norm': mean_absolute_error(y_true, y_pred_norm),
            'R2': r2_score(y_true_denorm, y_pred_denorm), 'RMSE': np.sqrt(mean_squared_error(y_true_denorm, y_pred_denorm)),
            'MAE': mean_absolute_error(y_true_denorm, y_pred_denorm),
            'Dir_Acc': dir_acc
        }
    return results

def prepare_jump_features(X_base, df_source, jump_cols):
    """Add jump features to feature matrix."""
    indices = X_base.index
    jump_feats = df_source.loc[indices, jump_cols].reset_index(drop=True)
    return pd.concat([X_base.reset_index(drop=True), jump_feats], axis=1)

# Prepare data
X_train_all, X_val_all, X_test_all, y_train, y_val, y_test, roll_train, roll_val, roll_test, actual_dvol = prepare_data_splits(
    train_df, val_df, test_df, all_features)

In [62]:
# =============================================================================
# LINEAR MODELS: TRAINING AND EVALUATION
# =============================================================================

linear_results = {}

# Train and evaluate each linear model
linear_specs = [
    ('OLS_NoLags', market_features),
    ('OLS_NoLags_Jumps', market_features + jump_feature_cols),
    ('HAR_RV', har_rv_features),
    ('OLS_WithLags', core_features),
    ('OLS_WithLags_Jumps', core_features + jump_feature_cols)
]

for name, features in linear_specs:
    model = LinearRegression()
    X_train, X_val, X_test = X_train_all[features], X_val_all[features], X_test_all[features]
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, roll_train, roll_val, roll_test, actual_dvol)
    linear_results[name] = {'model': model, 'features': features, 'metrics': metrics}

# Summary table
print("\n" + "="*90)
print("TEST SET PERFORMANCE (LINEAR MODELS)")
print("="*90)
print(f"{'Model':<20} {'Feats':>5} {'R²_norm':>9} {'R²':>9} {'RMSE':>8} {'MAE':>8} {'Dir%':>7}")
print("-"*90)
for name in ['OLS_NoLags', 'OLS_NoLags_Jumps', 'HAR_RV', 'OLS_WithLags', 'OLS_WithLags_Jumps']:
    m = linear_results[name]['metrics']['test']
    print(f"{name:<20} {len(linear_results[name]['features']):>5} {m['R2_norm']:>9.4f} {m['R2']:>9.4f} {m['RMSE']:>8.2f} {m['MAE']:>8.2f} {m['Dir_Acc']:>6.1f}%")



TEST SET PERFORMANCE (LINEAR MODELS)
Model                Feats   R²_norm        R²     RMSE      MAE    Dir%
------------------------------------------------------------------------------------------
OLS_NoLags               4    0.9470    0.9904     0.67     0.40   48.5%
OLS_NoLags_Jumps         8    0.9483    0.9906     0.66     0.40   49.3%
HAR_RV                   3    0.7136    0.9387     1.70     1.25   50.5%
OLS_WithLags             7    0.9469    0.9904     0.67     0.40   48.3%
OLS_WithLags_Jumps      11    0.9480    0.9906     0.67     0.40   49.2%


In [63]:
# =============================================================================
# TREE-BASED MODELS: FEATURE DEFINITIONS
# =============================================================================

# Feature definitions for tree-based models
tree_core_features = ['dvol_lag_1d_norm', 'dvol_lag_7d_norm', 'dvol_lag_30d_norm',
                      'transaction_volume_norm', 'network_activity_norm', 'nvrv_norm', 'dvol_rv_spread_norm']
market_features = ['transaction_volume_norm', 'network_activity_norm', 'nvrv_norm', 'dvol_rv_spread_norm']
jump_feature_cols = ['lee_mykland_jump', 'jump_magnitude_norm', 'days_since_jump_norm', 'jump_cluster_7d_norm']

# Helper function to add jump features to feature matrix
def prepare_jump_features(X_base, df_source, jump_cols):
    """Add jump features to base feature matrix.
    
    Args:
        X_base: Base feature DataFrame
        df_source: Source DataFrame containing jump columns
        jump_cols: List of jump column names to add
    
    Returns:
        Combined feature matrix with jump features
    """
    indices = X_base.index
    jump_feats = df_source.loc[indices, jump_cols].reset_index(drop=True)
    return pd.concat([X_base.reset_index(drop=True), jump_feats], axis=1)

In [64]:
# =============================================================================
# TREE-BASED MODELS: TRAINING AND EVALUATION
# =============================================================================

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

tree_results = {}

# Tree model specifications for consolidated training
# Format: (name, feature_list, is_xgb)
tree_model_specs = [
    ('RF_NoLag', market_features, False),
    ('RF_Lags', tree_core_features, False),
    ('RF_NoLag_Jumps', market_features + jump_feature_cols, False),
    ('RF_Lags_Jumps', tree_core_features + jump_feature_cols, False),
    ('XGB_NoLag', market_features, True),
    ('XGB_NoLag_Jumps', market_features + jump_feature_cols, True),
    ('XGB_Lags', tree_core_features, True),
    ('XGB_Lags_Jumps', tree_core_features + jump_feature_cols, True),
]

# Train all tree models using consolidated loop
for name, features, is_xgb in tree_model_specs:
    if is_xgb:
        # XGBoost hyperparameters per volatility forecasting literature
        # (Vrontos et al., 2021; Balaneji & Maringer, 2022)
        model = XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
    else:
        # Random Forest hyperparameters (Breiman, 2001)
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=4,
            random_state=42,
            n_jobs=-1
        )
    
    # Get the appropriate feature matrices based on model configuration
    has_lags = 'Lags' in name
    has_jumps = 'Jumps' in name
    
    if name.startswith('RF'):
        if has_jumps:
            X_train_mod = prepare_jump_features(
                X_train_all[tree_core_features if has_lags else market_features].copy(),
                train_df, jump_feature_cols)
            X_val_mod = prepare_jump_features(
                X_val_all[tree_core_features if has_lags else market_features].copy(),
                val_df, jump_feature_cols)
            X_test_mod = prepare_jump_features(
                X_test_all[tree_core_features if has_lags else market_features].copy(),
                test_df, jump_feature_cols)
        else:
            X_train_mod = X_train_all[tree_core_features if has_lags else market_features]
            X_val_mod = X_val_all[tree_core_features if has_lags else market_features]
            X_test_mod = X_test_all[tree_core_features if has_lags else market_features]
    else:  # XGB
        if has_jumps:
            X_train_mod = prepare_jump_features(
                X_train_all[tree_core_features if has_lags else market_features].copy(),
                train_df, jump_feature_cols)
            X_val_mod = prepare_jump_features(
                X_val_all[tree_core_features if has_lags else market_features].copy(),
                val_df, jump_feature_cols)
            X_test_mod = prepare_jump_features(
                X_test_all[tree_core_features if has_lags else market_features].copy(),
                test_df, jump_feature_cols)
        else:
            X_train_mod = X_train_all[tree_core_features if has_lags else market_features]
            X_val_mod = X_val_all[tree_core_features if has_lags else market_features]
            X_test_mod = X_test_all[tree_core_features if has_lags else market_features]
    
    # Train and evaluate
    model.fit(X_train_mod, y_train)
    tree_results[name] = {
        'model': model,
        'features': features,
        'metrics': evaluate_model(
            model, X_train_mod, y_train, X_val_mod, y_val,
            X_test_mod, y_test, roll_train, roll_val, roll_test, actual_dvol
        )
    }

# =============================================================================
# COMBINED SUMMARY
# =============================================================================
print("\n" + "="*95)
print("TEST SET PERFORMANCE (ALL MODELS)")
print("="*95)
print(f"{'Model':<22} {'Type':<8} {'Feats':>5} {'R²_norm':>9} {'R²':>9} {'RMSE':>8} {'MAE':>8} {'Dir%':>7}")
print("-"*95)

for name in ['OLS_NoLags', 'OLS_NoLags_Jumps', 'HAR_RV', 'OLS_WithLags', 'OLS_WithLags_Jumps']:
    m = linear_results[name]['metrics']['test']
    f = len(linear_results[name]['features'])
    print(f"{name:<22} {'Linear':<8} {f:>5} {m['R2_norm']:>9.4f} {m['R2']:>9.4f} {m['RMSE']:>8.2f} {m['MAE']:>8.2f} {m['Dir_Acc']:>6.1f}%")

for name in ['RF_NoLag', 'RF_Lags', 'RF_NoLag_Jumps', 'RF_Lags_Jumps', 'XGB_NoLag', 'XGB_NoLag_Jumps', 'XGB_Lags', 'XGB_Lags_Jumps']:
    m = tree_results[name]['metrics']['test']
    f = len(tree_results[name]['features'])
    print(f"{name:<22} {'Tree':<8} {f:>5} {m['R2_norm']:>9.4f} {m['R2']:>9.4f} {m['RMSE']:>8.2f} {m['MAE']:>8.2f} {m['Dir_Acc']:>6.1f}%")


TEST SET PERFORMANCE (ALL MODELS)
Model                  Type     Feats   R²_norm        R²     RMSE      MAE    Dir%
-----------------------------------------------------------------------------------------------
OLS_NoLags             Linear       4    0.9470    0.9904     0.67     0.40   48.5%
OLS_NoLags_Jumps       Linear       8    0.9483    0.9906     0.66     0.40   49.3%
HAR_RV                 Linear       3    0.7136    0.9387     1.70     1.25   50.5%
OLS_WithLags           Linear       7    0.9469    0.9904     0.67     0.40   48.3%
OLS_WithLags_Jumps     Linear      11    0.9480    0.9906     0.67     0.40   49.2%
RF_NoLag               Tree         4    0.9481    0.9905     0.67     0.40   48.7%
RF_Lags                Tree         7    0.9480    0.9906     0.66     0.40   48.3%
RF_NoLag_Jumps         Tree         8    0.9500    0.9909     0.65     0.39   49.0%
RF_Lags_Jumps          Tree        11    0.9490    0.9908     0.66     0.39   48.6%
XGB_NoLag              Tree  

## Cell 10: Tree-Based Model Specifications

### Methodology
Random Forest (Breiman, 2001) and XGBoost (Chen & Guestrin, 2016) are ensemble decision tree methods that capture non-linear relationships and feature interactions. Both use the same target variable (`dvol_norm.shift(-1)`) and preprocessing (720-hour rolling normalization) as linear models for fair comparison.

### Model Specifications

| Model | Hyperparameters |
|-------|-----------------|
| **Random Forest** | n_estimators=100, max_depth=10, min_samples_split=10, min_samples_leaf=4 |
| **XGBoost** | n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8 |

**Note:** XGBoost hyperparameters align with volatility forecasting literature (Vrontos et al., 2021; Balaneji & Maringer, 2022).

### Jump-Aware Modeling
XGBoost with jumps incorporates statistical jump detection features (Lee-Mykland, 2008), aligning with regime-switching literature (Zhang & Hua, 2025).

### Key References
- Breiman, L. (2001). *Random Forests*. *Machine Learning*, 45(1), 5-32.
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. *KDD*.
- Vrontos, I. et al. (2021). *Forecasting VIX with Machine Learning*. *Journal of Forecasting*.
- Balaneji, B., & Maringer, D. (2022). *Implied Volatility Forecasting with XGBoost*. *Quantitative Finance*.
- Zhang, L., & Hua, L. (2025). *High-Frequency Financial Data Analysis: A Survey*. *Mathematics*, 13(3), 347.

## Multi-Window Normalization Comparison

**Question**: Does the normalization window size affect model performance?

We test 4 different rolling window sizes:
- 72 hours (3 days)
- 168 hours (7 days)
- 336 hours (14 days)
- 720 hours (30 days) ← current default

**Total experiments**: 4 windows × 13 models = 52 runs


In [65]:
# =============================================================================
# MULTI-WINDOW EXPERIMENT CONFIGURATION
# =============================================================================

# Window sizes to test (hours)
WINDOW_SIZES = [72, 168, 336, 720]  # 3, 7, 14, 30 days

# Linear model specifications (same as original notebook)
linear_specs = [
    ('OLS_NoLags', ['transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread']),
    ('OLS_NoLags_Jumps', ['transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread',
                        'lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d']),
    ('HAR_RV', ['dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d']),
    ('OLS_WithLags', ['dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d',
                      'transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread']),
    ('OLS_WithLags_Jumps', ['dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d',
                            'transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread',
                            'lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d']),
]

# Tree model specifications
tree_specs = [
    ('RF_NoLag', 4, ['transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread'], False),
    ('RF_Lags', 7, ['dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d',
                   'transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread'], False),
    ('RF_NoLag_Jumps', 8, ['transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread',
                          'lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d'], False),
    ('RF_Lags_Jumps', 11, ['dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d',
                         'transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread',
                         'lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d'], False),
    ('XGB_NoLag', 4, ['transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread'], True),
    ('XGB_NoLag_Jumps', 8, ['transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread',
                          'lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d'], True),
    ('XGB_Lags', 7, ['dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d',
                   'transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread'], True),
    ('XGB_Lags_Jumps', 11, ['dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d',
                         'transaction_volume', 'network_activity', 'nvrv', 'dvol_rv_spread',
                         'lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d'], True),
]

def denormalize_predictions(y_pred_norm, df_norm, indices):
    """Convert normalized predictions back to raw scale using df_norm columns."""
    mean = df_norm.loc[indices, 'dvol_rolling_mean'].values
    std = df_norm.loc[indices, 'dvol_rolling_std'].values
    return y_pred_norm * std + mean

In [66]:
# =============================================================================
# MULTI-WINDOW EXPERIMENT FUNCTION
# =============================================================================

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def run_experiments_for_window(window_size):
    """Run all 13 models for a specific window size.
    
    Args:
        window_size: Normalization window size in hours
    
    Returns:
        Dictionary of model results with metrics including directional accuracy
    """
    
    print(f"\n{'='*60}")
    print(f"WINDOW: {window_size} hours ({window_size//24} days)")
    print(f"{'='*60}")
    
    # Build features list - MUST include 'dvol' for target creation
    all_features = ['dvol']  # Start with dvol so dvol_norm is created
    for spec in linear_specs:
        all_features.extend([f for f in spec[1] if f not in all_features])
    for spec in tree_specs:
        all_features.extend([f for f in spec[2] if f not in all_features])
    
    df_norm, scaling_params = apply_rolling_normalization(df, all_features, window=window_size)
    
    # Train/val/test split (60/20/20)
    n = len(df_norm)
    n_train = int(n * 0.60)
    n_val = int(n * 0.20)
    
    train_df = df_norm.iloc[:n_train].copy()
    val_df = df_norm.iloc[n_train:n_train + n_val].copy()
    test_df = df_norm.iloc[n_train + n_val:].copy()
    
    # Create targets (normalized next-period DVOL)
    train_df['target'] = train_df['dvol_norm'].shift(-1)
    val_df['target'] = val_df['dvol_norm'].shift(-1)
    test_df['target'] = test_df['dvol_norm'].shift(-1)
    
    # Store actual DVOL for directional accuracy calculation
    train_df['actual_dvol'] = train_df['dvol'].shift(-1)
    val_df['actual_dvol'] = val_df['dvol'].shift(-1)
    test_df['actual_dvol'] = test_df['dvol'].shift(-1)
    
    # Get all normalized feature names (excluding non-normalized columns)
    feature_cols_to_check = [f'{f}_norm' for f in all_features if f not in ['lee_mykland_jump', 'jump_indicator']]
    feature_cols_to_check.append('lee_mykland_jump')  # Add non-normalized jump column
    
    # Drop NaN from targets AND features
    train_df = train_df.dropna(subset=['target'] + feature_cols_to_check)
    val_df = val_df.dropna(subset=['target'] + feature_cols_to_check)
    test_df = test_df.dropna(subset=['target'] + feature_cols_to_check)
    
    y_train = train_df['target'].values
    y_val = val_df['target'].values
    y_test = test_df['target'].values
    
    # Store actual DVOL for directional accuracy
    actual_dvol_test = test_df['actual_dvol'].values
    
    print(f"Samples: {len(y_train)} train | {len(y_val)} val | {len(y_test)} test")
    
    results = {}
    
    # ---------------------------------------------------------------------
    # LINEAR MODELS (5)
    # ---------------------------------------------------------------------
    
    for name, features in linear_specs:
        # Build feature list with proper normalization
        final_features = []
        for f in features:
            if f in ['lee_mykland_jump', 'jump_indicator']:
                final_features.append(f)
            else:
                final_features.append(f'{f}_norm')
        
        # Prepare data
        X_train = train_df[final_features].values
        X_val = val_df[final_features].values
        X_test = test_df[final_features].values
        
        # Train model
        model = LinearRegression(fit_intercept=True)
        model.fit(X_train, y_train)
        
        # Predict
        y_pred_test = model.predict(X_test)
        
        # Denormalize predictions for evaluation (using test_df indices)
        y_pred_denorm = denormalize_predictions(y_pred_test, test_df, test_df.index)
        y_test_denorm = denormalize_predictions(y_test, test_df, test_df.index)
        
        # Calculate directional accuracy on denormalized values
        dir_acc = directional_accuracy(y_test_denorm, y_pred_denorm)
        
        # Metrics
        r2_norm = r2_score(y_test, y_pred_test)
        r2 = r2_score(y_test_denorm, y_pred_denorm)
        rmse = np.sqrt(mean_squared_error(y_test_denorm, y_pred_denorm))
        mae = mean_absolute_error(y_test_denorm, y_pred_denorm)
        
        results[name] = {
            'r2_norm': float(r2_norm),
            'r2': float(r2),
            'rmse': float(rmse),
            'mae': float(mae),
            'dir_acc': float(dir_acc),
            'features': len(features)
        }
    
    # ---------------------------------------------------------------------
    # TREE MODELS (8)
    # ---------------------------------------------------------------------
    
    for name, n_features, features, is_xgb in tree_specs:
        final_features = []
        for f in features:
            if f in ['lee_mykland_jump', 'jump_indicator']:
                final_features.append(f)
            else:
                final_features.append(f'{f}_norm')
        
        X_train = train_df[final_features].values
        X_val = val_df[final_features].values
        X_test = test_df[final_features].values
        
        if is_xgb:
            # XGBoost hyperparameters per volatility forecasting literature
            # (Vrontos et al., 2021; Balaneji & Maringer, 2022)
            model = XGBRegressor(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1
            )
        else:
            # Random Forest hyperparameters (Breiman, 2001)
            model = RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                min_samples_split=10,
                min_samples_leaf=4,
                random_state=42,
                n_jobs=-1
            )
        
        model.fit(X_train, y_train)
        y_pred_test = model.predict(X_test)
        
        # Denormalize
        y_pred_denorm = denormalize_predictions(y_pred_test, test_df, test_df.index)
        y_test_denorm = denormalize_predictions(y_test, test_df, test_df.index)
        
        # Calculate directional accuracy on denormalized values
        dir_acc = directional_accuracy(y_test_denorm, y_pred_denorm)
        
        results[name] = {
            'r2_norm': float(r2_score(y_test, y_pred_test)),
            'r2': float(r2_score(y_test_denorm, y_pred_denorm)),
            'rmse': float(np.sqrt(mean_squared_error(y_test_denorm, y_pred_denorm))),
            'mae': float(mean_absolute_error(y_test_denorm, y_pred_denorm)),
            'dir_acc': float(dir_acc),
            'features': n_features
        }
    
    return results

In [67]:
# Run experiments for all window sizes
all_window_results = {}

for window in WINDOW_SIZES:
    print(f"\nRunning experiments for window: {window}h ({window//24}d)...")
    all_window_results[window] = run_experiments_for_window(window)

print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETE")
print("="*80)



Running experiments for window: 72h (3d)...

WINDOW: 72 hours (3 days)
Samples: 23911 train | 8210 val | 8210 test

Running experiments for window: 168h (7d)...

WINDOW: 168 hours (7 days)
Samples: 23911 train | 8210 val | 8210 test

Running experiments for window: 336h (14d)...

WINDOW: 336 hours (14 days)
Samples: 23911 train | 8210 val | 8210 test

Running experiments for window: 720h (30d)...

WINDOW: 720 hours (30 days)
Samples: 23911 train | 8210 val | 8210 test

ALL EXPERIMENTS COMPLETE


In [68]:
# Compile comparison results
window_comparison = []

for window in WINDOW_SIZES:
    results = all_window_results[window]
    for name, metrics in results.items():
        window_comparison.append({
            'window_hours': window,
            'window_days': window // 24,
            'model': name,
            'features': metrics['features'],
            'r2_norm': metrics['r2_norm'],
            'r2': metrics['r2'],
            'rmse': metrics['rmse'],
            'mae': metrics['mae'],
            'dir_acc': metrics['dir_acc']
        })

comparison_df = pd.DataFrame(window_comparison)

# Full comparison table
print("\n" + "-"*100)
print("ALL MODELS - ALL WINDOWS (Ranking by R²)")
print("-"*100)
print(comparison_df.sort_values('r2', ascending=False).to_string(index=False))

# Best model by window
print("\n" + "="*100)
print("BEST MODEL BY WINDOW SIZE")
print("="*100)

for window in WINDOW_SIZES:
    window_data = comparison_df[comparison_df['window_hours'] == window]
    if not window_data.empty:
        best = window_data.loc[window_data['r2'].idxmax()]
        print(f"\n{window}h ({window//24}d): {best['model']}")
        print(f"  R²_norm: {best['r2_norm']:.4f} | R²: {best['r2']:.4f} | RMSE: {best['rmse']:.2f} | Dir%: {best['dir_acc']:.1f}%")

# Summary statistics
print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)

avg_r2_by_window = comparison_df.groupby('window_hours')['r2'].mean().to_dict()
best_r2_by_window = comparison_df.groupby('window_hours')['r2'].max().to_dict()
avg_dir_by_window = comparison_df.groupby('window_hours')['dir_acc'].mean().to_dict()

print(f"\nAverage R² by window:")
for w in WINDOW_SIZES:
    print(f"  {w}h ({w//24}d): {avg_r2_by_window[w]:.4f}")

print(f"\nBest R² by window:")
for w in WINDOW_SIZES:
    print(f"  {w}h ({w//24}d): {best_r2_by_window[w]:.4f}")

print(f"\nAverage Dir% by window:")
for w in WINDOW_SIZES:
    print(f"  {w}h ({w//24}d): {avg_dir_by_window[w]:.1f}%")

overall_best_window = max(avg_r2_by_window, key=avg_r2_by_window.get)
print(f"\nBest average window: {overall_best_window}h ({overall_best_window//24}d) with avg R² = {avg_r2_by_window[overall_best_window]:.4f}")

# Best directional accuracy by window
print("\n" + "="*100)
print("BEST DIRECTIONAL ACCURACY BY WINDOW SIZE")
print("="*100)

for window in WINDOW_SIZES:
    window_data = comparison_df[comparison_df['window_hours'] == window]
    if not window_data.empty:
        best_dir = window_data.loc[window_data['dir_acc'].idxmax()]
        print(f"\n{window}h ({window//24}d): {best_dir['model']}")
        print(f"  Dir%: {best_dir['dir_acc']:.1f}% | R²: {best_dir['r2']:.4f} | RMSE: {best_dir['rmse']:.2f}")


----------------------------------------------------------------------------------------------------
ALL MODELS - ALL WINDOWS (Ranking by R²)
----------------------------------------------------------------------------------------------------
 window_hours  window_days              model  features  r2_norm       r2     rmse      mae   dir_acc
           72            3    XGB_NoLag_Jumps         8 0.852028 0.993968 0.531397 0.326124 49.336095
           72            3     XGB_Lags_Jumps        11 0.849344 0.993646 0.545379 0.332544 49.104641
           72            3     RF_NoLag_Jumps         8 0.849924 0.993518 0.550843 0.332010 49.494457
           72            3      RF_Lags_Jumps        11 0.849180 0.993504 0.551439 0.332692 49.323913
           72            3           RF_NoLag         4 0.844891 0.992709 0.584216 0.341701 49.725911
           72            3           XGB_Lags         7 0.843587 0.992708 0.584247 0.342865 49.506639
           72            3          XGB_No